In [ ]:
# !pip install torch_geometric

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import gc
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import scipy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import torch_scatter
from typing import Dict, Tuple, List, Optional
import matplotlib.pyplot as plt
from torch_scatter import scatter_softmax, scatter_sum
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from torch.amp import autocast, GradScaler

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
system_size = 118

In [ ]:
datakit_data = np.load(f'/home/oarowolo/workfile/OPFData/data/Datakit/full_topology/pglib_opf_case{system_size}_ieee.npz', allow_pickle=True)

# Get all keys
print("Available keys in the dataset:")
for key in datakit_data.files:
    # Print the key and its array shape
    print(f"{key}: shape {datakit_data[key].shape}")

In [ ]:
datakit_data_bus = np.stack(datakit_data['bus_data'])
datakit_data_edge = np.stack(datakit_data['edge_data'])
datakit_data_gen = np.stack(datakit_data['gen_data'])
datakit_edge_index = np.stack(datakit_data['edge_index'])

In [ ]:
datakit_edge_index[0]

In [ ]:
def load_grid_data(grid_size:int):
   # Load the data
    data = np.load(f'/home/oarowolo/workfile/OPFData/data/OPFData/full_topology/{grid_size}bus_combined_dataset.npz')

    # Get all keys
    print("Available keys in the dataset:")
    for key in data.files:
        # Print the key and its array shape
        print(f"{key}: shape {data[key].shape}") 
        
        # Access grid input features
        grid_bus = data['grid_bus']  
        grid_generator = data['grid_generator']
        grid_load = data['grid_load']
        grid_shunt = data['grid_shunt']
        grid_ac_line_features = data['grid_ac_line_features']
        grid_transformer_features = data['grid_transformer_features']
        grid_ac_line_receivers = data['grid_ac_line_receivers']
        grid_ac_line_senders = data['grid_ac_line_senders']
        grid_transformer_senders = data['grid_transformer_senders']
        grid_transformer_receivers = data['grid_transformer_receivers']
        
        solution_bus = data['solution_bus']  
        solution_generator = data['solution_generator'] 
        solution_objective = data['metadata_objective']
        
        solution_objective = solution_objective.reshape(-1,1)

        branch_list = list(zip(grid_ac_line_senders[0], grid_ac_line_receivers[0]))
        transformer_list = list(zip(grid_transformer_senders[0], grid_transformer_receivers[0]))
        for k in transformer_list:
            branch_list.append(k)
            
        grid_generator_link_receivers = data['grid_generator_link_receivers']
        grid_load_link_receivers = data['grid_load_link_receivers']
        grid_shunt_link_receivers = data['grid_shunt_link_receivers']
        
        generator_indices = grid_generator_link_receivers[0]
        load_indices = grid_load_link_receivers[0]
        shunt_indices = grid_shunt_link_receivers[0]
        grid_transformer_senders = grid_transformer_senders[0]
        grid_transformer_receivers = grid_transformer_receivers[0]
        grid_ac_line_senders  = grid_ac_line_senders[0]
        grid_ac_line_receivers = grid_ac_line_receivers[0]
        
        return grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers, solution_bus, solution_generator, solution_objective, branch_list, generator_indices, load_indices, shunt_indices

In [ ]:
grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers,solution_bus, solution_generator, solution_objective, branch_list,  generator_indices,load_indices, shunt_indices = load_grid_data(system_size)

In [ ]:
## We have to use the load inputs, gen and bus outputs,variable gencosts, and rx from the Datakit data
## Keep the rest of the system features from OPFData since it's literally the same system
## We would use the rest of the system features to compute constraint satisfaction and so on.
## note that datakit does not automatically save the objective cost

In [ ]:
datakit_load = datakit_data_bus[:,:,:2]
datakit_solution_generator = datakit_data_bus[:,:,2:4]
datakit_solution_bus = datakit_data_bus[:,:,4:]
datakit_r_x = datakit_data_edge[:,:,:2]
datakit_gencosts = datakit_data_gen[:,:,:3]

In [ ]:
datakit_gencosts[0]

In [ ]:
datakit_solution_objectives = datakit_gencosts[:,:,0] * ((100*datakit_solution_generator[:,generator_indices,0])**2) + datakit_gencosts[:,:,1] * (100*datakit_solution_generator[:,generator_indices,0]) + datakit_gencosts[:,:,2]
datakit_solution_objectives = datakit_solution_objectives.sum(axis=1) 

In [ ]:
datakit_solution_objectives.mean()

In [ ]:
#what we want to do is separate the datakit_rx data into lines and transformers
#we can know the number of transformers from the second index of transformer features x, and select the last x branches in the branch list
# use the transformer branch tuples to find their indices in the datakit edge index data, use the indices to replace the original r_x in the OPF data
transformer_branches = branch_list[-grid_transformer_features.shape[1]:]
# Compare all edges at once
datakit_edge_list = datakit_edge_index[0]
transformer_branches = np.array(transformer_branches)
matches = (datakit_edge_list[0][:, None] == transformer_branches[:, 0]) & (datakit_edge_list[1][:, None] == transformer_branches[:, 1])
transformer_indices = np.where(matches.any(axis=1))[0]
line_indices = np.where(matches.any(axis=1) == False)[0]

In [ ]:
line_rx = datakit_r_x[:,line_indices,:]
transformer_rx = datakit_r_x[:,transformer_indices,:]

In [ ]:
grid_load = datakit_load[:,load_indices,:]
solution_bus = datakit_solution_bus  
solution_bus = solution_bus[:, :, ::-1]  # reverse the order of Va and Vm to match OPFData
solution_bus[:,:,0] = np.radians(solution_bus[:,:,0]) ## we also need to convert angle to radians for consistency
solution_generator = datakit_solution_generator[:,generator_indices,:]
solution_objective = datakit_solution_objectives

In [ ]:
def compute_gandb(edge_iputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
def get_B_matrix(N, edges, edge_weights):
    # Create a zero tensor of shape (N,N)
    B_matrix = torch.zeros((N, N)).to(torch.float64)
    
    # Unpack the edges into source and destination nodes
    sources, destinations = zip(*edges)
    
    # Use advanced indexing to place weights in the right spots
    B_matrix[sources, destinations] = edge_weights.squeeze()
    B_matrix[destinations, sources] = edge_weights.squeeze()
    return B_matrix

In [ ]:
def adjacency_to_laplacian(B_adj):

    # Ensure matrix is square
    assert B_adj.shape[0] == B_adj.shape[1], "Input must be square"

    # Copy to avoid modifying original
    B_laplacian = B_adj.copy()

    # Set diagonal as row sum of adjacency (i.e., degree)
    np.fill_diagonal(B_laplacian, -B_adj.sum(axis=1))

    return B_laplacian


In [ ]:
def another_effective_resistance_matrix(b_mat):
    """Compute effective resistance using a more efficient approach."""
    
    
    # Compute sparse Laplacian
    L = sp.csc_matrix(b_mat)
    num_nodes = L.shape[0]

    # Regularized Laplacian
    L_reg = L + 1e-10 * sp.eye(num_nodes)

    # Precompute factorization (Much faster than CG)
    L_solver = spla.factorized(L_reg)

    # Compute diagonal of pseudoinverse
    I = np.eye(num_nodes)
    L_plus_diag = np.array([L_solver(I[:, i])[i] for i in range(num_nodes)])

    # Compute resistance efficiently
    eff_res_matrix = np.zeros((num_nodes, num_nodes))

    for i in range(num_nodes):
        for j in range(i+1, num_nodes):
            e_ij = I[:, i] - I[:, j]
            x = L_solver(e_ij)  # Solve in one step

            resistance = np.dot(e_ij, x)
            eff_res_matrix[i, j] = resistance
            eff_res_matrix[j, i] = resistance  # Symmetric

    return eff_res_matrix, L_solver

In [ ]:
def effective_resistance_matrix(b_mat):
    """
    Computes the effective resistance matrix from a susceptance adjacency matrix.
    Converts it into a Laplacian first.
    
    Parameters:
    b_mat (ndarray): weighted laplacian matrix
    
    Returns:
    ndarray: Effective resistance matrix (N x N)
    """
    # Ensure symmetry
    L = b_mat

    n = L.shape[0]

    # Remove reference node (last row and column) to deal with singularity
    keep = np.arange(n - 1)
    L_reduced = L[np.ix_(keep, keep)]

    # Invert reduced Laplacian
    L_reduced_inv = np.linalg.inv(L_reduced)

    # Expand to full pseudoinverse
    L_plus = np.zeros((n, n))
    L_plus[np.ix_(keep, keep)] = L_reduced_inv

    # Project to orthogonal component (to make it true pseudoinverse)
    I = np.eye(n)
    ones = np.ones((n, n)) / n
    L_plus = (I - ones) @ L_plus @ (I - ones)

    # Compute resistance: R_ij = L^+_ii + L^+_jj - 2L^+_ij
    diag = np.diag(L_plus)
    R = diag[:, None] + diag[None, :] - 2 * L_plus
    return R

In [ ]:
# Vectorized version (more efficient for large matrices)
def compute_row_statistics_vectorized(resistance_matrix):
    """
    Vectorized computation of row statistics excluding diagonal elements.
    
    Parameters:
    resistance_matrix: numpy array of shape (N, N) with values between 0 and 1
    
    Returns:
    numpy array of shape (N, 5) with columns: [mean, median, std, max, min]
    """
    N = resistance_matrix.shape[0]
    
    # Create a mask to exclude diagonal elements
    mask = ~np.eye(N, dtype=bool)
    
    # Initialize result matrix
    stats_matrix = np.zeros((N, 5))
    
    # For each row, extract non-diagonal elements and compute statistics
    for i in range(N):
        row_no_diag = resistance_matrix[i, mask[i]]
        
        stats_matrix[i, 0] = np.mean(row_no_diag)
        stats_matrix[i, 1] = np.median(row_no_diag)
        stats_matrix[i, 2] = np.std(row_no_diag)
        stats_matrix[i, 3] = np.max(row_no_diag)
        stats_matrix[i, 4] = np.min(row_no_diag)
    
    return stats_matrix

In [ ]:
# for features we know are constant, we can simply choose the first index and multiply by the first dimension of Datakit data 

In [ ]:
samp_grid_bus = grid_bus[0]
samp_grid_generator = grid_generator[0]
samp_grid_shunt = grid_shunt[0]
samp_grid_ac_line_features = grid_ac_line_features[0]
samp_grid_transformer_features = grid_transformer_features[0]

In [ ]:
grid_bus = np.repeat(samp_grid_bus[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_generator = np.repeat(samp_grid_generator[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_shunt = np.repeat(samp_grid_shunt[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_ac_line_features = np.repeat(samp_grid_ac_line_features[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_transformer_features = np.repeat(samp_grid_transformer_features[np.newaxis, :, :], grid_load.shape[0], axis=0)

In [ ]:
## grid AC-line/transformer features are not constant, we should not act like they are!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
# now we replace the fixed generator costs and the fix rx with the actual variable costs and variable rx

In [ ]:
grid_generator[:,:,8:] = datakit_gencosts
grid_ac_line_features[:,:,4:6] = line_rx
grid_transformer_features[:,:,2:4] = transformer_rx

In [ ]:
grid_bus_list = []
grid_generator_list = []
grid_load_list = []
grid_shunt_list = []
grid_ac_line_features_list  = []
grid_transformer_features_list = []
solution_bus_list = []
solution_generator_list = []
# pe_list = []
pe_list = torch.load(f'Datakit_{system_size}_bus_fulltop_pe.pt')  ##we can load pe_list instead if it has been precomputed

In [ ]:
# for k in range(grid_bus.shape[0]):
for k in tqdm(range(grid_bus.shape[0]), desc="Data Processing Progress"):   
    
    grid_bus_k = grid_bus[k].astype(np.float32)
    grid_bus_list.append(grid_bus_k)
    grid_generator_k = grid_generator[k].astype(np.float32)
    grid_generator_list.append(grid_generator_k)
    grid_load_k = grid_load[k].astype(np.float32)
    grid_load_list.append(grid_load_k)
    grid_shunt_k = grid_shunt[k].astype(np.float32)
    grid_shunt_list.append(grid_shunt_k)
    
    solution_bus_k = solution_bus[k].astype(np.float32)
    solution_bus_list.append(solution_bus_k)
    solution_generator_k = solution_generator[k].astype(np.float32)
    solution_generator_list.append(solution_generator_k)


    # grid_transformer_senders_list.append(grid_transformer_sender)
    # grid_transformer_receivers_list.append(grid_transformer_receiver)
    # grid_ac_line_senders_list.append(grid_ac_line_sender)
    # grid_ac_line_receivers_list.append(grid_ac_line_receiver)

    grid_transformer_features_k = grid_transformer_features[k].astype(np.float32)
    grid_transformer_features_list.append(grid_transformer_features_k)
    grid_ac_line_features_k = grid_ac_line_features[k].astype(np.float32)
    grid_ac_line_features_list.append(grid_ac_line_features_k)

    ################################ this is where we create the positional encoding stuff
    # edge_inputs = np.zeros((len(branch_list),11))
    # edge_inputs[:grid_ac_line_features_k.shape[0],:9] = grid_ac_line_features_k  # rearranging edge inputs to align for transformers and transmission lines
    # edge_inputs[grid_ac_line_features_k.shape[0]:,:2] =  grid_transformer_features_k[:,:2]
    # edge_inputs[grid_ac_line_features_k.shape[0]:,2:4] =  grid_transformer_features_k[:,9:]
    # edge_inputs[grid_ac_line_features_k.shape[0]:,4:9] =  grid_transformer_features_k[:,2:7]
    # edge_inputs[grid_ac_line_features_k.shape[0]:,9:] =  grid_transformer_features_k[:,7:9]
    # edge_inputs[:grid_ac_line_features_k.shape[0],9:10] = 1.0

    # edge_g, edge_b = compute_gandb(edge_inputs)
    # B_weighted = get_B_matrix(system_size, branch_list,torch.tensor(edge_b))
    # b_mat = np.array(B_weighted)
    # B_lap = adjacency_to_laplacian(b_mat)
    # e_R = effective_resistance_matrix(B_lap)
    # raw_PE = compute_row_statistics_vectorized(e_R)
    # bus_pe = torch.tensor(raw_PE,dtype=torch.float)
    # pe_list.append([bus_pe])

In [ ]:
# flatter_pe_list = [j[0] for j in pe_list]
# torch.save(flatter_pe_list, f'Datakit_{system_size}_bus_fulltop_pe.pt')

In [ ]:
batch_size = 128

In [ ]:
from torch_geometric.data import HeteroData

def create_grid_hetero_data(
    grid_bus,                    
    grid_generator,              
    grid_load,                   
    grid_shunt,                  
    grid_ac_line_features,       
    grid_transformer_features,   
    grid_ac_line_senders,        
    grid_ac_line_receivers,      
    grid_transformer_senders,    
    grid_transformer_receivers,  
    generator_indices,          
    load_indices,                
    shunt_indices,               
    solution_bus,                
    solution_generator,
    pe,
    batch_idx=0
):
    
    # Create a new HeteroData instance
    data = HeteroData()
    
    # Process a single batch if specified, otherwise we'd need to handle batching differently
    if batch_idx is not None:
        # Extract features for the specified batch
        bus_features = torch.tensor(grid_bus[batch_idx], dtype=torch.float)
        generator_features = torch.tensor(grid_generator[batch_idx], dtype=torch.float)
        load_features = torch.tensor(grid_load[batch_idx], dtype=torch.float)
        shunt_features = torch.tensor(grid_shunt[batch_idx], dtype=torch.float)


        bus_pe = pe[batch_idx].to(torch.float)
        
        ac_line_features = torch.tensor(grid_ac_line_features[batch_idx], dtype=torch.float)
        transformer_features = torch.tensor(grid_transformer_features[batch_idx], dtype=torch.float)
        
        # Extract solution values for the specified batch
        bus_solutions = torch.tensor(solution_bus[batch_idx], dtype=torch.float)
        generator_solutions = torch.tensor(solution_generator[batch_idx], dtype=torch.float)
        
        # Add node features
        data['bus'].x = bus_features
        data['generator'].x = generator_features
        data['load'].x = load_features
        data['shunt'].x = shunt_features

        #create global IDs for nodes to make indexing of transformer easier
        data['bus'].graph_id = torch.full((bus_features.shape[0],), batch_idx, dtype=torch.long)
        data['generator'].graph_id = torch.full((generator_features.shape[0],), batch_idx, dtype=torch.long)
        data['load'].graph_id = torch.full((load_features.shape[0],), batch_idx, dtype=torch.long)
        data['shunt'].graph_id = torch.full((shunt_features.shape[0],), batch_idx, dtype=torch.long)

        ####use indices specially
        g_indices = generator_indices
        s_indices = shunt_indices
        l_indices = load_indices

        #Add node positional encoding
        data['bus'].pe = bus_pe
        data['generator'].pe = bus_pe[g_indices]  ## added arbitrary values to distinguish buses from special nodes
        data['load'].pe = bus_pe[l_indices]
        data['shunt'].pe = bus_pe[s_indices]
        
        # Add solution values as target values (y)
        data['bus'].y = bus_solutions
        data['generator'].y = generator_solutions
        
        # Add edge indices and features for AC lines (bus to bus)
        senders = torch.tensor(grid_ac_line_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_ac_line_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'ac_line', 'bus'].edge_index = edge_index
        data['bus', 'ac_line', 'bus'].edge_attr = ac_line_features
        
        # Add edge indices and features for transformers (bus to bus)
        senders = torch.tensor(grid_transformer_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_transformer_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'transformer', 'bus'].edge_index = edge_index
        data['bus', 'transformer', 'bus'].edge_attr = transformer_features
        
        # Add pseudo-edges from generators to buses
        gen_to_bus = torch.tensor(generator_indices.flatten(), dtype=torch.long)
        gen_indices = torch.arange(len(gen_to_bus), dtype=torch.long)
        gen_edge_index = torch.stack([gen_indices, gen_to_bus], dim=0)
        data['generator', 'connects_to', 'bus'].edge_index = gen_edge_index
        data['generator', 'connects_to', 'bus'].edge_attr = torch.ones((len(gen_to_bus), 3))
        
        # Add pseudo-edges from loads to buses
        load_to_bus = torch.tensor(load_indices.flatten(), dtype=torch.long)
        load_index = torch.arange(len(load_to_bus), dtype=torch.long)
        load_edge_index = torch.stack([load_index, load_to_bus], dim=0)
        data['load', 'connects_to', 'bus'].edge_index = load_edge_index
        data['load', 'connects_to', 'bus'].edge_attr = torch.ones((len(load_to_bus), 3))
        
        # Add pseudo-edges from shunts to buses
        shunt_to_bus = torch.tensor(shunt_indices.flatten(), dtype=torch.long)
        shunt_index = torch.arange(len(shunt_to_bus), dtype=torch.long)
        shunt_edge_index = torch.stack([shunt_index, shunt_to_bus], dim=0)
        data['shunt', 'connects_to', 'bus'].edge_index = shunt_edge_index
        data['shunt', 'connects_to', 'bus'].edge_attr = torch.ones((len(shunt_to_bus), 3))
    
    else:
        # Handle all batches (would require batching approach)
        raise NotImplementedError("Processing all batches at once is not implemented in this example")
    
    return data


def create_dataloader(
    grid_bus,
    grid_generator,
    grid_load,
    grid_shunt,
    grid_ac_line_features,
    grid_transformer_features,
    grid_ac_line_senders,
    grid_ac_line_receivers,
    grid_transformer_senders,
    grid_transformer_receivers,
    generator_indices,
    load_indices,
    shunt_indices,
    solution_bus,
    solution_generator,
    pe,
    batch_size=batch_size,
    shuffle = True
):
    
    # Create a list of HeteroData objects
    dataset = []
    
    for i in range(len(grid_bus)):  # Process up to 1000 samples for this example
        data = create_grid_hetero_data(
            grid_bus, 
            grid_generator,
            grid_load,
            grid_shunt,
            grid_ac_line_features,
            grid_transformer_features,
            grid_ac_line_senders,
            grid_ac_line_receivers,
            grid_transformer_senders,
            grid_transformer_receivers,
            generator_indices,
            load_indices,
            shunt_indices,
            solution_bus,
            solution_generator,
            pe,
            batch_idx=i
        )
        dataset.append(data)
    
    # Create a DataLoader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,num_workers=min(8, torch.get_num_threads()))
    
    return loader

In [ ]:
data_len = grid_load.shape[0]

In [ ]:
def train_val_test_split(data_len, train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = data_len
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    return train_indices, val_indices, test_indices

In [ ]:
train_indices, val_indices, test_indices = train_val_test_split(data_len)

In [ ]:
train_grid_bus =  list(grid_bus_list[i] for i in train_indices)
train_grid_generator=  list(grid_generator_list[i] for i in train_indices)
train_grid_load=  list(grid_load_list[i] for i in train_indices)
train_grid_shunt=  list(grid_shunt_list[i] for i in train_indices)
train_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in train_indices)
train_grid_transformer_features=  list(grid_transformer_features_list[i] for i in train_indices)
train_solution_bus=  list(solution_bus_list[i] for i in train_indices)
train_solution_generator=  list(solution_generator_list[i] for i in train_indices)
train_bus_pe = [pe_list[i] for i in train_indices]

In [ ]:
validate_grid_bus =  list(grid_bus_list[i] for i in val_indices)
validate_grid_generator=  list(grid_generator_list[i] for i in val_indices)
validate_grid_load=  list(grid_load_list[i] for i in val_indices)
validate_grid_shunt=  list(grid_shunt_list[i] for i in val_indices)
validate_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in val_indices)
validate_grid_transformer_features=  list(grid_transformer_features_list[i] for i in val_indices)
validate_solution_bus=  list(solution_bus_list[i] for i in val_indices)
validate_solution_generator=  list(solution_generator_list[i] for i in val_indices)
validate_bus_pe = list(pe_list[i] for i in val_indices)

In [ ]:
test_grid_bus =  list(grid_bus_list[i] for i in test_indices)
test_grid_generator=  list(grid_generator_list[i] for i in test_indices)
test_grid_load=  list(grid_load_list[i] for i in test_indices)
test_grid_shunt=  list(grid_shunt_list[i] for i in test_indices)
test_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in test_indices)
test_grid_transformer_features=  list(grid_transformer_features_list[i] for i in test_indices)
test_solution_bus=  list(solution_bus_list[i] for i in test_indices)
test_solution_generator=  list(solution_generator_list[i] for i in test_indices)
test_bus_pe = list(pe_list[i] for i in test_indices)

In [ ]:
train_loader= create_dataloader(train_grid_bus,
                                train_grid_generator,
                                train_grid_load,
                                train_grid_shunt,
                                train_grid_ac_line_features,
                                train_grid_transformer_features,
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                train_solution_bus,
                                train_solution_generator,
                                train_bus_pe,
                                batch_size=batch_size,
                                shuffle = True)

In [ ]:
val_loader=   create_dataloader(validate_grid_bus,
                                validate_grid_generator,
                                validate_grid_load,
                                validate_grid_shunt,
                                validate_grid_ac_line_features,
                                validate_grid_transformer_features,
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                validate_solution_bus,
                                validate_solution_generator,
                                validate_bus_pe,
                                batch_size=batch_size,
                                shuffle = True)

In [ ]:
test_loader = create_dataloader(test_grid_bus,
                                test_grid_generator,
                                test_grid_load,
                                test_grid_shunt,
                                test_grid_ac_line_features,
                                test_grid_transformer_features,
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                test_solution_bus,
                                test_solution_generator,
                                test_bus_pe,
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
# del train_grid_bus  
# del train_grid_generator 
# del train_grid_load 
# del train_grid_shunt 
# del train_grid_ac_line_features 
# del train_grid_transformer_features 

In [ ]:
del validate_grid_bus  
del validate_grid_generator 
del validate_grid_load 
del validate_grid_shunt 
del validate_grid_ac_line_features 
del validate_grid_transformer_features 

In [ ]:
class MLP(torch.nn.Module):
    """Multi-Layer Perceptron with optional LayerNorm and LeakyReLU."""

    def __init__(self, input_size, hidden_size, output_size, layers,
                 layernorm=True, use_leaky=False):
        super().__init__()
        modules = []
        for i in range(layers):
            modules.append(torch.nn.Linear(
                input_size if i == 0 else hidden_size,
                output_size if i == layers - 1 else hidden_size,
            ))
            if i != layers - 1:
                modules.append(torch.nn.ReLU())
            if use_leaky:
                modules.append(torch.nn.LeakyReLU(negative_slope=0.02))
        if layernorm:
            modules.append(torch.nn.LayerNorm(output_size))
        self.network = torch.nn.Sequential(*modules)
        self.reset_parameters()

    def reset_parameters(self):
        for layer in self.network:
            if isinstance(layer, torch.nn.Linear):
                layer.weight.data.normal_(0, 1 / math.sqrt(layer.in_features))
                layer.bias.data.fill_(0)

    def forward(self, x):
        return self.network(x)

In [ ]:
### This is a special new implementation that follows exactly from the GPS paper.
# from performer_pytorch import SelfAttention
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn.attention import PerformerAttention

class HeteroPerformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attn = PerformerAttention(
            channels=hidden_dim,
            heads=num_heads
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Post-attention MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
    def forward(self, x_dict, xm_dict, batch_dict):
            # Flatten all node types
            flat_x, flat_xm, flat_batch = [], [], []
            slices = {}
            offset = 0
    
            for ntype in x_dict:
                x = x_dict[ntype]
                xm = xm_dict[ntype]
                b = batch_dict[ntype]  # batch indices for each node
    
                slices[ntype] = slice(offset, offset + x.size(0))
                flat_x.append(x)
                flat_xm.append(xm)
                flat_batch.append(b)
                offset += x.size(0)
    
            x_all = torch.cat(flat_x, dim=0)         # [N, D]
            xm_all = torch.cat(flat_xm, dim=0)       # [N, D]
            global_batch_all = torch.cat(flat_batch, dim=0) # [N]

            _, batch_all = torch.unique(global_batch_all, return_inverse=True)
            sorted_indices = torch.argsort(batch_all) # resort batch index to be ascending but that means I have to sort the xs myself too
            batch_all_sorted = batch_all[sorted_indices]
            x_all_sorted = x_all[sorted_indices]
            xm_all_sorted = xm_all[sorted_indices]
            # Convert to [B, N_max, D] and mask            
            x_dense, mask = to_dense_batch(x_all_sorted, batch_all_sorted)   # [B, N, D], [B, N]
            
            # Apply masked Performer attention
            x_attn = self.attn(x_dense, mask=mask)              # [B, N, D]
            # Residual + Norm
            xt_out = self.norm1(x_dense + x_attn)          
            # Unpad: [real_nodes, D]
            xt_out = xt_out[mask]
    
            x_comb = self.norm2(xt_out + xm_all_sorted)
            x_final = self.mlp(x_comb)

            # dont forget to give x_final it unsorted arrangement
            unsorted_x = torch.empty_like(x_final)
            unsorted_x[sorted_indices] = x_final
            x_final = unsorted_x
            # x_final = x_final[sorted_indices]

            # Unflatten by slice
            return {ntype: x_final[slices[ntype]] for ntype in x_dict.keys()}


In [ ]:
class HeteroInteractionNetwork(nn.Module):
    """
    One message-passing step over a heterogeneous graph.
    Physical edges (AC lines, transformers) include edge features in messages;
    pseudo-edges (gen/load/shunt → bus) use only node features.
    """

    def __init__(self, node_types, edge_types, physical_edge_types, hidden_size, layers):
        super().__init__()
        self.physical_edge_types = physical_edge_types

        self.edge_updaters = nn.ModuleDict()
        for src, rel, dst in edge_types:
            key       = f"{src}_{rel}_{dst}"
            edge_type = (src, rel, dst)
            in_dim    = hidden_size * 3 if edge_type in physical_edge_types else hidden_size * 2
            self.edge_updaters[key] = MLP(in_dim, hidden_size, hidden_size, layers)

        self.node_updaters = nn.ModuleDict({
            nt: MLP(hidden_size * 2, hidden_size, hidden_size, layers)
            for nt in node_types
        })

    def forward(self, x_dict, edge_indices_dict, edge_features_dict):
        updated_edge_features = {}
        aggregated_messages   = {nt: torch.zeros_like(feat) for nt, feat in x_dict.items()}

        for edge_type, edge_index in edge_indices_dict.items():
            src_type, rel_type, dst_type = edge_type
            edge_key = f"{src_type}_{rel_type}_{dst_type}"
            src, dst = edge_index
            x_i = x_dict[dst_type][dst]
            x_j = x_dict[src_type][src]

            if edge_type in self.physical_edge_types:
                edge_feature = edge_features_dict[edge_type]
                updated_edge = self.edge_updaters[edge_key](
                    torch.cat((x_i, x_j, edge_feature), dim=-1)
                )
                updated_edge = updated_edge + edge_feature   # residual
                updated_edge_features[edge_type] = updated_edge
            else:
                updated_edge = self.edge_updaters[edge_key](
                    torch.cat((x_i, x_j), dim=-1)
                )
                updated_edge_features[edge_type] = updated_edge

            # Cast before scatter (needed for mixed-precision)
            aggregated_messages[dst_type] = aggregated_messages[dst_type].to(updated_edge.dtype)
            aggregated_messages[src_type] = aggregated_messages[src_type].to(updated_edge.dtype)
            aggregated_messages[dst_type] = torch_scatter.scatter_add(
                updated_edge, dst, dim=0, out=aggregated_messages[dst_type])
            aggregated_messages[src_type] = torch_scatter.scatter_add(
                updated_edge, src, dim=0, out=aggregated_messages[src_type])

        updated_nodes = {}
        for nt, x in x_dict.items():
            node_update = self.node_updaters[nt](
                torch.cat((x, aggregated_messages[nt]), dim=-1)
            )
            updated_nodes[nt] = x + node_update  # residual

        return updated_nodes, updated_edge_features

In [ ]:
class HeteroInteractGNN(torch.nn.Module):
    """
    Heterogeneous GPS-style GNN for ACOPF.
    Architecture identical to the full-topology version — variable graph sizes
    are handled naturally by PyG batching.
    """

    def __init__(self, hidden_size=256, n_mp_layers=5, bus_features=4,
                 gen_features=11, load_features=2, shunt_features=2,
                 ac_line_features=9, transformer_features=11,
                 connects_to_features=3, output_dim=2):
        super().__init__()

        self.node_types = ['bus', 'generator', 'load', 'shunt']
        self.edge_types = [
            ('bus', 'ac_line',     'bus'),
            ('bus', 'transformer', 'bus'),
            ('generator', 'connects_to', 'bus'),
            ('load',      'connects_to', 'bus'),
            ('shunt',     'connects_to', 'bus'),
        ]
        self.physical_edge_types = [
            ('bus', 'ac_line',     'bus'),
            ('bus', 'transformer', 'bus'),
        ]

        self.node_encoders = nn.ModuleDict({
            'bus':       MLP(bus_features,   hidden_size, hidden_size - 5, 2),
            'generator': MLP(gen_features,   hidden_size, hidden_size - 5, 2),
            'load':      MLP(load_features,  hidden_size, hidden_size - 5, 2),
            'shunt':     MLP(shunt_features, hidden_size, hidden_size - 5, 2),
        })
        self.global_attn_layers = nn.ModuleList([
            HeteroPerformerLayer(hidden_size) for _ in range(n_mp_layers)
        ])
        self.edge_encoders = nn.ModuleDict({
            'ac_line':     MLP(ac_line_features,      hidden_size, hidden_size, 2),
            'transformer': MLP(transformer_features,  hidden_size, hidden_size, 2),
        })
        self.n_mp_layers = n_mp_layers
        self.layers = torch.nn.ModuleList([
            HeteroInteractionNetwork(
                self.node_types, self.edge_types, self.physical_edge_types, hidden_size, 2
            )
            for _ in range(n_mp_layers)
        ])
        self.node_decoders = nn.ModuleDict({
            'bus':       MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
            'generator': MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
        })

    def forward(self, data):
        x_dict = {}
        for nt in self.node_types:
            if hasattr(data[nt], 'x'):
                enc = self.node_encoders[nt](data[nt].x)
                x_dict[nt] = torch.cat([enc, data[nt].pe], dim=-1)

        edge_feature_dict = {}
        for src, rel, dst in self.edge_types:
            et = (src, rel, dst)
            if (et in self.physical_edge_types and et in data.edge_types
                    and hasattr(data[et], 'edge_attr')):
                edge_feature_dict[et] = self.edge_encoders[rel](data[et].edge_attr)

        edge_index_dict = {
            et: data[et].edge_index
            for et in self.edge_types
            if et in data.edge_types and hasattr(data[et], 'edge_index')
        }
        batch_dict = {nt: data[nt].batch for nt in x_dict}

        for i in range(self.n_mp_layers):
            xm_dict, edge_feature_dict = self.layers[i](x_dict, edge_index_dict, edge_feature_dict)
            x_dict = self.global_attn_layers[i](x_dict, xm_dict, batch_dict)

        return {
            'bus':torch.sigmoid(self.node_decoders['bus'](x_dict['bus'])),
            'generator':torch.sigmoid(self.node_decoders['generator'](x_dict['generator'])),
        }

In [ ]:
## wandb set-up
api_key = 'my_key'
wandb.login(key=api_key)

In [ ]:
def convert_voltage_bounds(model_input):

    num_nodes = model_input.shape[0]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    thetamin =  torch.tensor([-2.00]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([2.00]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin),dim=1)


    return bounds_up, bounds_down

In [ ]:
def convert_power_bounds(model_input):

    num_nodes = model_input.shape[0]

    pmin = model_input[:,2:3]

    pmax = model_input[:,3:4]

    
    qmin = model_input[:,5:6]

    qmax = model_input[:,6:7]

    bounds_up = torch.concat((pmax, qmax), dim=1)
    bounds_down = torch.concat((pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))

#     for batch in tqdm(trainloader, desc="Training"):
    for batch in trainloader:
    
        optimizer.zero_grad(set_to_none=True)
        
        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        # model.redraw_projection.redraw_projections()
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
            
    
            # voltage_loss = criterion(voltages, batch['bus'].y)
            # power_loss = criterion(powers, batch['generator'].y)
            # loss = voltage_loss + power_loss
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
            loss = criterion(combined_targets, combined_outputs)
    
            # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()


            
            
        total_loss += loss.item()
        
    return total_loss / len(trainloader)

In [ ]:
# Example training step
def validate_model(model, val_loader):
    
    model.eval()
    total_loss = 0
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))
    
#     for batch in tqdm(trainloader, desc="Training"):
    for batch in val_loader:
        
        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
    
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
    
            loss = criterion(combined_targets, combined_outputs)
     
        total_loss += loss.item()
        
    return total_loss / len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))
    
    total_loss = 0.0
    voltage_predictions = []
    voltage_targets = []
    power_predictions = []
    power_targets = []
    
    for batch in testloader:

        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
            
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
            loss = criterion(combined_targets, combined_outputs)
        

        total_loss += loss.item()

        # Store predictions and targets for overall metrics
        voltage_predictions.append(voltages.cpu())
        voltage_targets.append(batch['bus'].y.cpu())

        power_predictions.append(powers.cpu())
        power_targets.append(batch['generator'].y.cpu())

    
    return total_loss / len(testloader), voltage_predictions, voltage_targets, power_predictions, power_targets

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"Datakit_{system_size}_HybridHeteroGNN_5_256_PQVT_special",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GNN",
      "dataset": "Datakit",
      "epochs": 100,
      })

In [ ]:
model = HeteroInteractGNN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, fused=True, weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
training_losses = []
validation_losses = []
best_valid_loss = float('inf')
early_stop_thresh = 100
best_epoch = -1
best_model_state = None
num_epochs = 100
for epoch in tqdm(range(num_epochs), desc="Training Progress"):
    train_loss = train_model(model, train_loader, optimizer)
    valid_loss = validate_model(model, val_loader)
    training_losses.append(train_loss)
    validation_losses.append(valid_loss)

    wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

    scheduler.step(train_loss)

    if epoch % 10 == 0:
      print(f'Epoch: {epoch}')
      print(f'\tTrain Loss: {train_loss:.4f}')
      print(f'\t Val. Loss: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      best_model_state = deepcopy(model.state_dict())

plt.subplots(figsize=(5,3))
plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
plt.legend()
plt.title(f'GNN Training and Validation loss',fontsize = 15)
plt.xlabel('Epochs',fontsize = 12)
plt.ylabel('MSE Loss',fontsize = 12)
plt.semilogy()

training_losses=np.array(training_losses)
validation_losses=np.array(validation_losses)

model.load_state_dict(best_model_state)
model.eval()

In [ ]:
torch.save(model.state_dict(), f"{system_size}_bus_HeteroGNN_5_256_PQVT_Datakit_special.pth")
wandb.save(f"{system_size}_bus_HeteroGNN_5_256_PQVT_Datakit_special.pth")  # Upload to WandB

In [ ]:
test_loss, v_predictions, v_targets,p_predictions, p_targets = test_model(model, test_loader)

In [ ]:
print('loss on test data is ', test_loss)

In [ ]:
v_predictions = torch.cat(v_predictions, dim=0)
v_targets = torch.cat(v_targets, dim=0)

In [ ]:
p_predictions = torch.cat(p_predictions, dim=0)
p_targets = torch.cat(p_targets, dim=0)

In [ ]:
v_predictions = v_predictions.reshape(-1,system_size, 2)
v_targets = v_targets.reshape(-1, system_size, 2)

In [ ]:
p_predictions = p_predictions.reshape(-1,generator_indices.shape[0], 2)
p_targets = p_targets.reshape(-1, generator_indices.shape[0], 2)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(v_predictions[:,:,0],v_targets[:,:,0])
voltage_magnitude_loss = calc_loss(v_predictions[:,:,1],v_targets[:,:,1])

In [ ]:
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(p_predictions[:,:,0],p_targets[:,:,0])
reactive_power_loss = calc_loss(p_predictions[:,:,1],p_targets[:,:,1])
print('average gen active power error is  ', active_power_loss)
print('average gen reactive power error is  ', reactive_power_loss)

In [ ]:
load_demand = torch.zeros(grid_load[test_indices].shape[0], system_size, grid_load[test_indices].shape[-1])
load_demand[:,load_indices,:] = torch.tensor(grid_load[test_indices]).float()

In [ ]:
# branch_list = list(zip(grid_ac_line_senders.flatten(), grid_ac_line_receivers.flatten()))
# transformer_list = list(zip(grid_transformer_senders.flatten(), grid_transformer_receivers.flatten()))
# for k in transformer_list:
#     branch_list.append(k)

In [ ]:
edge_inputs = np.zeros((grid_bus.shape[0],len(branch_list),11))

edge_inputs[:,:grid_ac_line_features.shape[1],:9] = grid_ac_line_features  # rearranging edge inputs to align for transformers and transmission lines
edge_inputs[:,grid_ac_line_features.shape[1]:,:2] =  grid_transformer_features[:,:,:2]
edge_inputs[:,grid_ac_line_features.shape[1]:,2:4] =  grid_transformer_features[:,:,9:]
edge_inputs[:,grid_ac_line_features.shape[1]:,4:9] =  grid_transformer_features[:,:,2:7]
edge_inputs[:,grid_ac_line_features.shape[1]:,9:] =  grid_transformer_features[:,:,7:9]
edge_inputs[:,:grid_ac_line_features.shape[1],9:10] = 1.0

In [ ]:
edge_test = edge_inputs[test_indices]

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,:,4:5]
    line_x = edge_inputs[:,:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
edge_g, edge_b = compute_gandb(edge_test)

In [ ]:
edge_test = torch.tensor(edge_test)

In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=2)

In [ ]:
def convert_to_complex_rectangle_3D(tensor_3d):
    # Extract angle and magnitude
    tensor_mag = tensor_3d[:,:,0:1]  # In radians
    tensor_angle = tensor_3d[:,:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=2)

In [ ]:
conductance_susceptance = np.concatenate((edge_g, edge_b), axis=2)
conductance_susceptance = torch.tensor(conductance_susceptance)
charging_susceptance = torch.zeros_like(conductance_susceptance)
charging_susceptance[:,:,1:] =  edge_test[:,:,2:3].to('cpu')
Tij = edge_test[:,:,9:].to('cpu')
Tij[:,:grid_ac_line_features.shape[1],0:1] = 1.0
Tij_rec = convert_to_complex_rectangle_3D(Tij)

In [ ]:
def calculate_only_branch_flows(
    demand: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    voltage: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    branches: list,  # list of 20 tuples (from_bus, to_bus)
    Yks: torch.Tensor,  # shape (batch_size,14,2) [real, imag] shunt admittance
    Yij: torch.Tensor,  # shape (batch_size,20,2) [real, imag] branch admittance
    Yijc: torch.Tensor,  # shape (batch_size,20,2) [real, imag] branch charging admittance
    Tij: torch.Tensor,  # shape (batch_size,20,2) [real, imag] transformation ratio
) -> torch.Tensor:
    batch_size = demand.shape[0]
    num_nodes = voltage.shape[1]
    
    # Helper function for batched complex multiplication
    def complex_mult_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1],
            a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0]
        ], dim=-1)

    # Helper function for batched complex conjugate
    def complex_conj_batch(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[..., 0], -x[..., 1]], dim=-1)

    # Helper function for batched complex division
    def complex_div_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[..., 0]**2 + b[..., 1]**2
        return torch.stack([
            (a[..., 0] * b[..., 0] + a[..., 1] * b[..., 1]) / denominator,
            (a[..., 1] * b[..., 0] - a[..., 0] * b[..., 1]) / denominator
        ], dim=-1)

    # Initialize generator power tensor
    generator_power = torch.zeros_like(demand)
    
    # Calculate shunt power terms for each node (vectorized)
    v_mag_sq = torch.sum(voltage**2, dim=-1, keepdim=True)  # shape: (batch_size, 14, 1)
    v_mag_sq = torch.cat([v_mag_sq, torch.zeros_like(v_mag_sq)], dim=-1)  # shape: (batch_size, 14, 2)
    
    # Yks is already batched
    shunt_power = complex_mult_batch(complex_conj_batch(Yks), v_mag_sq)
    
    # Pre-allocate branch flows dictionary with tensors
    branch_flows = {}
    
    # Calculate branch flows (vectorized)
    for idx, (i, j) in enumerate(branches):
        
        # Get complex voltage at both ends
        vi = voltage[:, i]  # shape: (batch_size, 2)
        vj = voltage[:, j]  # shape: (batch_size, 2)
        
        # First term calculations
        vi_mag_sq = torch.sum(vi**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
        vi_mag_sq = torch.cat([vi_mag_sq, torch.zeros_like(vi_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
        tij_mag_sq = torch.sum(Tij[:, idx]**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
        tij_mag_sq_tensor = torch.cat([tij_mag_sq, torch.zeros_like(tij_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
        vi_over_tij_sq = complex_div_batch(vi_mag_sq, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance (already batched)
        Y_total = Yij[:, idx] + Yijc[:, idx]  # shape: (batch_size, 2)
        
        term1 = complex_mult_batch(complex_conj_batch(Y_total), vi_over_tij_sq)
        
        # Second term calculations
        vivj = complex_mult_batch(vi, complex_conj_batch(vj))
        term2 = complex_mult_batch(
            complex_conj_batch(Yij[:, idx]),
            complex_div_batch(vivj, Tij[:, idx])
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        
        # Reverse flow calculations
        vj_mag_sq = torch.sum(vj**2, dim=-1, keepdim=True)
        vj_mag_sq = torch.cat([vj_mag_sq, torch.zeros_like(vj_mag_sq)], dim=-1)
        
        term1_ji = complex_mult_batch(complex_conj_batch(Y_total), vj_mag_sq)
        vjvi = complex_mult_batch(complex_conj_batch(vi), vj)
        term2_ji = complex_mult_batch(
            complex_conj_batch(Yij[:, idx]),
            complex_div_batch(vjvi, complex_conj_batch(Tij[:, idx]))
        )
        
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
    
    
    # Aggregate generator power for each node (vectorized)
    for i in range(num_nodes):
        for index, (from_bus, to_bus) in enumerate(branches):
            if from_bus == i:
                generator_power[:, i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[:, i] += branch_flows[(to_bus, from_bus, index)]
    
    generator_power += demand + shunt_power
    
    return generator_power, branch_flows

In [ ]:
complex_v = convert_to_complex_voltage(v_predictions)
complex_v = complex_v.to(torch.float32)

In [ ]:
load_demand = load_demand.to(torch.float32)

In [ ]:
Yks = torch.zeros(grid_shunt[test_indices].shape[0], system_size, grid_shunt[test_indices].shape[-1]).to(torch.float32)

Yks[:,shunt_indices,:] = torch.tensor(grid_shunt[test_indices])

Yks = Yks[:,:, [1, 0]]

In [ ]:
injection_balance,branch_flows = calculate_only_branch_flows(load_demand.to('cpu'),complex_v.to('cpu'),branch_list,Yks,conductance_susceptance,charging_susceptance,Tij_rec)

In [ ]:
def compute_optimality(test_inputs, test_outputs, test_objective):

    test_inputs = test_inputs.cpu()
    test_outputs = test_outputs.cpu()
    test_objective = test_objective.cpu()

    c2 = test_inputs[:,:,8:9] 
    c1 = test_inputs[:,:,9:10] 
    c0 = test_inputs[:,:,10:11] 

    # Get relevant output dimensions (zero-indexed)
    p_gens = test_outputs[:,:,0:1] # select on Pgs for generators
  

    print('the shape of c2 is ', c2.shape)
    print('the shape of p_gen is ', p_gens.shape)
    
    # Compute node-wise metrics
    system_metrics = c2 * ((100 * p_gens) ** 2) + c1 * (100*p_gens) + c0


    model_obj = torch.sum(system_metrics, dim=1).flatten()

    print('the shape of test objective is ', test_objective.shape)
    print('the shape of model objective is ', model_obj.shape)

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {test_objective.mean()}')

    optimality_gap = (model_obj / test_objective) * 100
  
    return optimality_gap.mean()

In [ ]:
test_obj = torch.tensor(solution_objective[test_indices])
test_gen_inputs = torch.tensor(grid_generator[test_indices])

In [ ]:
opt_gap = compute_optimality(test_gen_inputs, p_predictions, test_obj)

In [ ]:
print('optimality gap now is ', opt_gap)

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles is a NumPy array
    angles = np.asarray(angles)
    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
##### time to evaluate constrain satisfactions

In [ ]:
predicted_angle_differences = np.zeros((v_predictions.shape[0], len(branch_list)))
predicted_angles = v_predictions[:,:,0]

for j in range(v_predictions.shape[0]):
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_list)
    predicted_angle_differences[j] = angle_differences

In [ ]:
true_angle_differences = np.zeros((v_targets.shape[0], len(branch_list)))
true_angles = v_targets[:,:,0]

for j in range(v_targets.shape[0]):
    angle_differences = calculate_angle_differences(true_angles[j],branch_list)
    true_angle_differences[j] = angle_differences

In [ ]:
predicted_angle_differences = torch.tensor(predicted_angle_differences)
true_angle_differences = torch.tensor(true_angle_differences)

In [ ]:
# voltage angle difference bound 
angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max())
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
test_bus_inputs = torch.tensor(grid_bus[test_indices])

In [ ]:
## voltage magnitude bound
vmin = test_bus_inputs[:,:,2:3].to('cpu')

vmax = test_bus_inputs[:,:,3:4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - v_predictions[:,:,1:2], min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(v_predictions[:,:,1:2] - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max())
print('average voltage magnitude violation is : ',vmag_violations.mean())

In [ ]:
test_generator_inputs = torch.tensor(grid_generator[test_indices])

In [ ]:
# Gen active power bounds 
pmin = test_generator_inputs[:,:,2:3].to('cpu')

pmax = test_generator_inputs[:,:,3:4].to('cpu')

p_gens = p_predictions[:,:,0:1]

lower_pgen_violations = torch.clamp(pmin - p_gens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(p_gens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max())
print('average active power violation is : ',pgen_violations.mean())

In [ ]:
# Gen reactive power bounds 
qmin = test_generator_inputs[:,:,5:6].to('cpu')

qmax = test_generator_inputs[:,:,6:7].to('cpu')

q_gens = p_predictions[:,:,1:2]

lower_qgen_violations = torch.clamp(qmin - q_gens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(q_gens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max())
print('average reactive power violation is : ',qgen_violations.mean())

In [ ]:
forward_keys = [(i, j, index) for index, (i,j) in enumerate(branch_list)]
reverse_keys = [(j, i, index) for index, (i,j) in enumerate(branch_list)]

In [ ]:
len(forward_keys)

In [ ]:
len(reverse_keys)

In [ ]:
forward_branch_flows = {key: branch_flows[key] for key in forward_keys if key in branch_flows}
reverse_branch_flows = {key: branch_flows[key] for key in reverse_keys if key in branch_flows}

In [ ]:
forward_power_flows = [tensor.unsqueeze(dim=1) for tensor in forward_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
forward_power_flows = torch.cat(forward_power_flows, dim=1)

In [ ]:
reverse_power_flows = [tensor.unsqueeze(dim=1) for tensor in reverse_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
reverse_power_flows = torch.cat(reverse_power_flows, dim=1)

In [ ]:
# compute the power magnitudes
# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 

    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)

In [ ]:
# Branch flow bounds in forward direction
long_term_line_rating = edge_test[0][:,6:7].to('cpu')
branch_flow_limit = long_term_line_rating.tile((p_predictions.shape[0],1,1))
forward_branch_flow = forward_flow_magnitude
forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound
print('max forward power flow violation is : ',forward_flow_violations.max())
print('average  forward power flow violation is : ',forward_flow_violations.mean())

In [ ]:
# Branch flow bounds in reverse direction

reverse_branch_flow = reverse_flow_magnitude

reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound

print('max reverse power flow violation is : ', reverse_flow_violations.max() )
print('average reverse power flow violation is : ', reverse_flow_violations.mean() )

In [ ]:
gen_power = torch.zeros(p_predictions.shape[0],system_size,2)
# gen_power[:,generator_indices,:] = p_targets
for i in range(p_predictions.shape[0]):
    gen_power[i, :, :].index_add_(0, torch.tensor(generator_indices), p_predictions[i, :, :])

In [ ]:
## Evaluate power balance constraint violations


real_power_balance_mismatches = injection_balance[:,:,0] - gen_power[:,:,0]


print('max active power balance mismatch is : ', real_power_balance_mismatches.max())
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean())

In [ ]:
reactive_power_balance_mismatches = injection_balance[:,:,1] - gen_power[:,:,1]


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max())
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns=["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", forward_flow_violations.max())
model_metrics_table.add_data("average forward power flows violation", forward_flow_violations.mean())
model_metrics_table.add_data("max reverse power flows violation", reverse_flow_violations.max())
model_metrics_table.add_data("average reverse power flows violation", reverse_flow_violations.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
torch.cuda.empty_cache()
gc.collect()